# [8.1] Activation Patching Refresher - Exercises

Build the local activation-patching contract: answer logit-diff metrics, clean-into-corrupt activation patching, recovered fractions, localization reports, and top-vs-random controls.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter8_automated_circuits"
section = "part1_activation_patching_refresher"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_activation_patching_refresher.tests as tests

GT_TIER = "GT-1"
EXERCISE_ID = "8.1.activation_patching_refresher"
EXPECTED_RUNTIME = "20-35 minutes for exercises; about 1 minute for CUDA preflight"
REQUIRES_GPU = False

## Logit-Diff Metric

Compute the scalar positive-minus-negative answer logit difference used for clean, corrupt, and patched runs.

In [ ]:
def answer_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


tests.test_answer_logit_diff_validates_token_ids(answer_logit_diff)

## Patching Mechanics

Patch exactly one clean activation slice into a corrupt activation tensor without mutating the inputs.

In [ ]:
def patch_activation_slice(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    *,
    component_index: int,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_patch_activation_slice_replaces_one_component_without_mutating_inputs(
    patch_activation_slice,
)

## Recovery Scores

Normalize patched metrics by the clean-corrupt gap, then rank each patched component by recovered fraction.

In [ ]:
@dataclass(frozen=True)
class PatchingRecoveryReport:
    clean_metric: float
    corrupt_metric: float
    patched_metric: float
    recovered_fraction: float
    passes_recovery: bool


@dataclass(frozen=True)
class ActivationPatchingSweep:
    patch_scores: t.Tensor
    best_index: int
    best_score: float


def recovery_fraction(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metric: float,
) -> float:
    raise NotImplementedError()


def patching_recovery_report(
    clean_logits: t.Tensor,
    corrupt_logits: t.Tensor,
    patched_logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
    min_recovered_fraction: float = 0.5,
) -> PatchingRecoveryReport:
    raise NotImplementedError()


def activation_patching_sweep(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metrics: t.Tensor,
) -> ActivationPatchingSweep:
    raise NotImplementedError()


tests.test_patching_recovery_report_and_sweep_normalize_by_clean_corrupt_gap(
    patching_recovery_report,
    activation_patching_sweep,
    recovery_fraction,
)

## Localization And Controls

Check top-k overlap with known target components and compare the top score against same-size random controls.

In [ ]:
@dataclass(frozen=True)
class PatchingLocalizationReport:
    top_indices: tuple[int, ...]
    target_indices: tuple[int, ...]
    topk_overlap: float
    localizes_target: bool


@dataclass(frozen=True)
class RandomPatchControlReport:
    top_patch_score: float
    random_patch_score: float
    top_beats_random: bool


def patching_localization_report(
    patch_scores: t.Tensor,
    target_indices: list[int],
    *,
    top_k: int = 2,
    min_overlap: float = 0.5,
) -> PatchingLocalizationReport:
    raise NotImplementedError()


def random_patch_control_report(
    patch_scores: t.Tensor,
    random_indices: list[int],
    *,
    top_k: int = 2,
) -> RandomPatchControlReport:
    raise NotImplementedError()


tests.test_localization_and_random_controls_require_top_components_to_win(
    patching_localization_report,
    random_patch_control_report,
)

## Combined Contract

After each helper passes, compose the CPU smoke report.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
